# CAPSTONE PROJEKTAS: Prekybos centro pardavimų ir klientų elgsenos analizė

**Duomenų rinkinys:** Supermarket Sales Dataset
**Tikslas:** Atlikti išsamią duomenų analizę, siekiant suprasti klientų pirkimo įpročius, įvertinti pardavimų tendencijas ir sukurti prognozavimo modelius.

**Analizės etapai:**
1. Duomenų paruošimas ir apžvalga.
2. Aprašomoji analizė (EDA).
3. Pasikliautiniai intervalai.
4. Hipotezių testavimas.
5. Regresinė analizė.
6. Laiko eilučių analizė.

In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import statsmodels.api as sm
from scipy import stats

# Nustatome vizualizacijų stilių
sns.set_theme(style="whitegrid")
print("Bibliotekos sėkmingai užkrautos!")

Bibliotekos sėkmingai užkrautos!


In [2]:
# Užkrauname duomenis
df = pd.read_csv('SuperMarketAnalysis.csv')

# Pažiūrime į pirmas 5 eilutes
df.head()

FileNotFoundError: [Errno 2] No such file or directory: 'SuperMarketAnalysis.csv'

In [ ]:
print(f"Duomenų rinkinys turi {df.shape[0]} eilučių ir {df.shape[1]} stulpelių.")

In [ ]:
print("--- Duomenų info ---")
print(df.info())

In [ ]:
print("\n--- Trūkstamos reikšmės ---")
print(df.isnull().sum())

In [ ]:
print("\n--- Aprašomoji statistika (skaitiniai kintamieji) ---")
display(df.describe())

## 2. Aprašomoji analizė (EDA)
Šioje dalyje analizuosime pagrindinį kintamąjį `Sales` (pardavimus) ir jo ryšius su kitais faktoriais.

In [ ]:
plt.figure(figsize=(10, 5))
sns.histplot(df['Sales'], kde=True, color='teal')
plt.title('Pardavimų sumos (Sales) pasiskirstymas')
plt.xlabel('Pardavimų suma')
plt.ylabel('Dažnis')
plt.show()

# Papildomai išvedame skaitines reikšmes pagal reikalavimus
print(f"Vidurkis: {df['Sales'].mean():.2f}")
print(f"Mediana: {df['Sales'].median():.2f}")
print(f"Standartinis nuokrypis: {df['Sales'].std():.2f}")

**Įžvalgos:**
Pardavimų sumų (`Sales`) pasiskirstymas yra asimetriškas su teigiama nuokrypa (angl. *positive skew*). Matome, kad dauguma pirkimų yra santykinai nedideli (iki 400), o pirkimų, viršijančių 800, pasitaiko rečiau. Tai rodo, kad prekybos centre dominuoja vidutiniai pirkinių krepšeliai. Regresinei analizei tai gali reikšti, kad paklaidos gali būti nevisiškai normaliosios.

In [ ]:
plt.figure(figsize=(10, 6))
sns.boxplot(x='City', y='Sales', data=df, palette='viridis')
plt.title('Pardavimų sumos pasiskirstymas skirtinguose miestuose')
plt.xlabel('Miestas')
plt.ylabel('Pardavimų suma')
plt.show()

**Įžvalgos:**
Pardavimų medianos visuose trijuose miestuose (Yangon, Naypyitaw, Mandalay) yra labai panašios. Tai rodo, kad geografinė lokacija nedaro kritinės įtakos vieno pirkimo sumai. Visuose miestuose matome keletą išskirčių (angl. *outliers*) – itin didelių pirkimų, tačiau bendra duomenų sklaida (kvartiliai) išlieka stabili.

In [ ]:
# Tikriname tik skaitinių kintamųjų koreliaciją
plt.figure(figsize=(12, 8))
# Pasirenkame tik skaitinius stulpelius
numeric_df = df.select_dtypes(include=[np.number])
sns.heatmap(numeric_df.corr(), annot=True, cmap='coolwarm', fmt=".2f")
plt.title('Kintamųjų tarpusavio koreliacijos matrica')
plt.show()

**Įžvalgos ir kritinė pastaba:**
Koreliacijų matrica atskleidžia tobulą koreliaciją (1.00) tarp `Sales` ir tokių kintamųjų kaip `Tax 5%`, `cogs` bei `gross income`. 
* **Svarbu:** Tai yra loginė duomenų savybė, nes šie kintamieji apskaičiuojami tiesiogiai iš pardavimo sumos. 
* **Veiksmas:** Siekiant išvengti multikolinearumo problemos (kurią vėliau patikrinsime VIF testu), šie kintamieji nebus įtraukiami į regresijos modelį kaip prognozuojantys požymiai.

## 3. Pasikliautiniai intervalai
Šioje dalyje apskaičiuosime 95 % pasikliautinį intervalą klientų įvertinimų (`Rating`) vidurkiui. Tai padės suprasti, kokiame rėžyje tikėtina rasti tikrąjį visų klientų pasitenkinimo vidurkį.

In [ ]:
import scipy.stats as stats

# Duomenys
ratings = df['Rating']
mean_rating = ratings.mean()
std_error = stats.sem(ratings) # Standartinė paklaida
confidence_level = 0.95

# Apskaičiuojame intervalą naudojant t-pasiskirstymą (pagal paskaitų pavyzdžius)
ci = stats.t.interval(confidence_level, len(ratings)-1, loc=mean_rating, scale=std_error)

print(f"Įvertinimų (Rating) vidurkis: {mean_rating:.2f}")
print(f"95% pasikliautinis intervalas: ({ci[0]:.2f}, {ci[1]:.2f})")

**Interpretacija:**
Mes esame 95 % tikri, kad tikrasis visos populiacijos (visų prekybos centro klientų) įvertinimų (`Rating`) vidurkis yra tarp **6,97** ir **7,08**. Kadangi šis intervalas yra siauras, galime daryti išvadą, kad mūsų gautas vidurkis (6,97) yra tikslus ir patikimas įvertis. Verslo požiūriu tai rodo, kad klientų pasitenkinimas yra stabilus ir vidutiniškai siekia apie 7 balus iš 10.

## 4. Hipotezių testavimas
Tikriname, ar lojalumo programa turi įtakos pirkimo sumai. 
**H0:** Vidutinė pirkimo suma tarp 'Member' ir 'Normal' klientų nesiskiria.
**Ha:** Vidutinė pirkimo suma tarp šių grupių skiriasi.

In [ ]:
# Atskiriame grupes
member_sales = df[df['Customer type'] == 'Member']['Sales']
normal_sales = df[df['Customer type'] == 'Normal']['Sales']

# Atliekame nepriklausomų imčių t-testą
t_stat, p_val = stats.ttest_ind(member_sales, normal_sales)

print(f"t-statistika: {t_stat:.4f}")
print(f"p-reikšmė: {p_val:.4f}")

# Rezultato interpretacija
alpha = 0.05
if p_val < alpha:
    print("Išvada: Atmetame nulinę hipotezę (H0). Skirtumas tarp grupių yra statistiškai reikšmingas.")
else:
    print("Išvada: Nepavyko atmesti nulinės hipotezės (H0). Statistiškai reikšmingo skirtumo tarp pirkimo sumų nėra.")

**Statistinė interpretacija:**
Atliktas t-testas parodė, kad p-reikšmė yra 0.0611. Kadangi p > 0.05, mes priimame nulinę hipotezę. Tai reiškia, kad lojalumo kortelės turėjimas šiuo konkrečiu laikotarpiu neturėjo didelės įtakos vieno pirkimo sumai. 

**Verslo įžvalga:**
Nors nariai gali pirkti dažniau, jų vienkartinio krepšelio dydis yra panašus į atsitiktinių pirkėjų. Rekomenduojama peržiūrėti lojalumo programos naudas, siekiant paskatinti didesnės vertės pirkimus.

## 5. Analitinė dalis
### A. Regresinė analizė

Šiame etape sukursime daugybinės tiesinės regresijos modelį, siekdami suprasti, kokie veiksniai daro įtaką pirkimo sumai (`Sales`). 
**Metodika:**
1. Pašaliname kintamuosius, kurie sukelia tobulą multikolinearumą (`gross income`, `cogs`, `Tax 5%`).
2. Sukuriame *dummy* kintamuosius kategoriniams duomenims (`drop_first=True`).
3. Padaliname duomenis į mokymo (80%) ir testavimo (20%) rinkinius.

In [ ]:
from sklearn.model_selection import train_test_split
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
from statsmodels.stats.outliers_influence import variance_inflation_factor

# 1. Pasirenkame kintamuosius
features = ['Branch', 'Customer type', 'Gender', 'Rating', 'Product line']
X = df[features]
y = df['Sales']

# 2. Dummy variables (Pagal 6 paskaitą: drop_first=True ir astype(float))
X = pd.get_dummies(X, drop_first=True).astype(int)

# 3. Pridedame konstantą (Statsmodels reikalavimas)
X = sm.add_constant(X)

# 4. Duomenų padalinimas (Pagal 5 paskaitą)
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

print("Duomenys paruošti. Mokymo imtis:", X_train.shape[0], "eilučių.")
X.head()

In [ ]:
# Modelio fit'inimas
model = sm.OLS(y_train, X_train).fit()

# Išvedame suvestinę
print(model.summary())

# Multikolinearumo patikra (VIF)
vif_data = pd.DataFrame()
vif_data["Feature"] = X.columns
vif_data["VIF"] = [variance_inflation_factor(X.values, i) for i in range(len(X.columns))]
print("\n--- VIF rezultatai ---")
display(vif_data)

### Regresijos rezultatų interpretacija ir įžvalgos

**1. Modelio kokybė ir patikimumas:**
* **R-squared (0,019):** Modelis paaiškina apie 1,9 % pirkimo sumos (`Sales`) variacijos. Nors skaičius nedidelis, jis rodo, kad pirkimo suma prekybos centre yra labiau priklausoma nuo individualių krepšelio savybių, o ne nuo kliento demografinių požymių.
* **VIF testas:** Visos VIF reikšmės yra artimos 1 (maksimali apie 1,7), o tai yra gerokai žemiau kritinės ribos (5 arba 10). Tai patvirtina, kad modelyje **nėra multikolinearumo problemų** ir koeficientai yra stabilūs.

**2. Statistiškai reikšmingi kintamieji:**
* Vienintelis kintamasis, kurio p-reikšmė yra mažesnė už 0,05, yra **Gender_Male (p = 0,013)**. 
* **Koeficientas (-43,55):** Tai rodo, kad, laikant kitus kintamuosius nekintamais, vyrų vidutinis pirkimo krepšelis yra vidutiniškai **43,55 vienetais mažesnis** nei moterų (kuri yra bazinė kategorija).

**3. Kiti faktoriai:**
* **Branch, Customer type ir Product line** p-reikšmės viršija 0,05, todėl šiame duomenų rinkinyje jie neturi statistiškai reikšmingos įtakos vienkartinio pirkimo sumai.
* **Rating (p = 0,241):** Klientų pasitenkinimo įvertinimas taip pat tiesiogiai nekoreliuoja su išleista suma.

**Verslo rekomendacija:** Kadangi moterys vidutiniškai išleidžia daugiau, rinkodaros kampanijos, orientuotos į didesnės vertės krepšelius, galėtų būti tikslingiau nukreiptos į moterų segmentą, tuo tarpu vyrų segmentui gali reikėti skatinimo priemonių vidutiniam krepšeliui didinti.

In [ ]:
# Prognozės testiniams duomenims
y_pred = model.predict(X_test)

# Metrikos
mae = mean_absolute_error(y_test, y_pred)
rmse = np.sqrt(mean_squared_error(y_test, y_pred))
r2_test = r2_score(y_test, y_pred)

print(f"MAE: {mae:.2f}")
print(f"RMSE: {rmse:.2f}")
print(f"R-squared (Test): {r2_test:.4f}")